> Copyright (c) 2022 DFKI GmbH - All Rights Reserved
> 
> Written by Michael Fürst <Michael.Fuerst@dfki.de>, October 2022

# Example: NuScenes - Loading from Tars

> Here we load nuscenes from the tars into datapipes.

## Inspecting the metadata

First we create the base datapipeline creator to inspect the metadata in order to choose our parameters.


In [ ]:
from datapipes.base_pipeline import BasePipelineCreator

TAR_DATASET = "/home/fuerst/Desktop/Datasets/nuscenes/keyframes_mini"
factory = BasePipelineCreator(TAR_DATASET)

Our goal is to create a datapipe, to do this we need to know the following parameters:
* subset
* shuffle_buffer_size
* component_group_filter
* shuffle_shards

To choose correctly it makes sense to learn more about our shards first.
Let's print:
* available subsets
* the number of tars for each subset
* available component groups
* stats for each component group

In [ ]:
for subset in factory.get_subsets():
    print(f"Subset: {subset}")
    tars = factory.get_tar_files_for_subsets(subsets=[subset])
    print(f"Num Shards: {len(tars)}")
    groups = factory.get_component_groups(subset)
    print(f"Num Slices: {len(tars) / len(groups)}")
    shardlen = factory.get_average_shard_sample_count(subset)
    print(f"Avg Shard Len: {shardlen}")
    stats = factory.get_component_groups_stats(subset)
    for group in groups:
        if subset == "train":
            print(f"  Component Group: {group} {stats[group]}")

Now we can create a datapipe by specifying our parameters.
We want to use the `mini_train` subset and use all components except radar.
As we use it for training, we will shuffle the shards.

Choosing the `shuffle_buffer_size` is more difficult. In experiments simulating the randomness of the dataset we learned we want roughly `sequence_length * batch_size * 10`. However, in this example we will use only 40, to keep memory consumption on the local machine low.

In [ ]:
batch_size = 1
components = [x for group, stats in factory.get_component_groups_stats("mini_train").items() if group != "radar" for x in stats["all_components"]]

pipe = factory.create_datapipe(
    subsets=["mini_train"], 
    shuffle_buffer_size=int(factory.get_average_shard_sample_count("mini_train") * batch_size),
    shuffle_shards=True,
    components=components
)

## Add a decoder

Now the pipe will give us raw data, we need to decode it to use it. For this we can use decoders.

In [ ]:
from datapipes.utils.dispatcher import Dispatcher
from datapipes.decoders.image_decoder import ImageDecoder
from datapipes.decoders.pointcloud_decoder import PointcloudDecoder

decoders = {
    'CAM_FRONT': ImageDecoder('torchrgb'),
    'CAM_FRONT_RIGHT': ImageDecoder('torchrgb'),
    'CAM_FRONT_LEFT': ImageDecoder('torchrgb'),
    'CAM_BACK': ImageDecoder('torchrgb'),
    'CAM_BACK_LEFT': ImageDecoder('torchrgb'),
    'CAM_BACK_RIGHT': ImageDecoder('torchrgb'),
    'LIDAR_TOP': PointcloudDecoder(output_torch=True),
}

pipe = pipe.map(Dispatcher(decoders))

## Benchmark the pipeline

We benchmark the read speed of the pipeline to see if our additions after this stage are efficient or not.

In [ ]:
import time
from tqdm import tqdm
import matplotlib.pyplot as plt

start = time.time()
start1 = start
N = 200
FPS = []
for i, sample in enumerate(tqdm(pipe, desc="Benchmarking")):
    stop = time.time()
    FPS.append(1.0 / ((stop - start1)))
    start1 = stop
    if i >= N - 1:
        break
stop = time.time()

plt.plot(FPS)
plt.show()

FPS = 1.0 / ((stop - start) / N)
print(FPS)

## Inject Annotations from Global File

Having the data decoded we want to inject our global annotations now.
Annotations can be read using the nuScenes dataset implementation, but we will not load images from there.

In [ ]:
from nuscenes import NuScenes
from nuscenes.utils.splits import train, val, test

class InjectNuscenesAnnotations(object):
    def __init__(self, version, dataroot) -> None:
        self.dataset = NuScenes(version=version, dataroot=dataroot, verbose=False)
    
    def __call__(self, sample):
        out = {}
        for fname, data in sample:
            out["sample_token"] = fname.split(".")[-3]
            out["scene"] = fname.split("/")[-1].split(".")[0]
            out[fname.split(".")[-2]] = data
        
        sample = self.dataset.get('sample', out["sample_token"])
        out["annotations"] = []
        for token in sample['anns']:
            out["annotations"].append(self.dataset.get('sample_annotation', token))

        return out

RAW_DATASET="/ds-av/public_datasets/nuscenes/raw"
pipe = pipe.map(InjectNuscenesAnnotations("v1.0-mini", RAW_DATASET))

## Test the pipe

With the pipe created, we want to test if we can get a sample from it.
To do this simply iterate over it with a for loop.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for i, sample in enumerate(pipe):
    print(f"Sample: {i}")
    plot_idx = 1
    plt.figure(figsize=(14,6))
    for component_name, component_data in sample.items():
        if "CAM" in component_name:
            plt.subplot(1, 6, plot_idx)
            plt.title(component_name)
            plt.imshow(component_data.permute(1, 2, 0))
            plot_idx += 1
        else:
            print("# " + component_name)
            if isinstance(component_data, np.ndarray):
                print(component_data.shape)
            else:
                text = str(component_data)
                append = ""
                if len(text) > 400:
                    append = " ..."
                print(text[:400] + append)
    plt.show()
    if i >= 2:
        break

Benchmark again to see if the speed changed.

In [ ]:
import time
from tqdm import tqdm

start = time.time()
start1 = start
N = 200
FPS = []
for i, sample in enumerate(tqdm(pipe, desc="Benchmarking")):
    stop = time.time()
    FPS.append(1.0 / ((stop - start1)))
    start1 = stop
    if i >= N - 1:
        break
stop = time.time()

plt.plot(FPS)
plt.show()

FPS = 1.0 / ((stop - start) / N)
print(FPS)

As we can see the injection of the annotations produces no additional load. Since they are stored in single database file which is loaded into RAM once.